# NeuroFusionNet — Paper Walkthrough

**Paper:** Akbar et al., *"NeuroFusionNet: a hybrid EEG feature fusion framework for accurate and explainable Alzheimer's Disease detection"*  
Scientific Reports (2025) 15:43742 · DOI: https://doi.org/10.1038/s41598-025-28070-x

This notebook walks through each component of NeuroFusionNet, connecting paper
sections to code. All inputs are random tensors — **no data or GPU needed**.

---

**Reduced dimensions used throughout** (paper values shown in comments):
- `C = 4` channels (paper: 18)
- `L = 64` segment samples (paper: 1000)
- `BATCH = 2`
- `dfinal = 20` PCA features (paper: ≈60–80)

The architecture is identical — only sizes are smaller for CPU execution.

In [ ]:
import math, sys, warnings
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

# Toy dimensions (paper values in comments)
BATCH = 2
C     = 4      # paper: 18 channels
L     = 64     # paper: 1000 samples per segment
D2    = 32     # paper: d2=256 (CNN output)
DFINAL= 20     # paper: dfinal≈60-80 (post-PCA)
N_CLS = 3      # AD / FTD / CN

print('Setup complete. PyTorch:', torch.__version__)

---
## 1. Handcrafted Feature Extraction (§Feature extraction, Table 10)

> *"Each EEG segment was passed through a multi-domain analytical pipeline involving
> statistical, spectral, wavelet, and entropy-based descriptors. Features were computed
> channel-wise and concatenated into a unified vector f_hand^(i,j) ∈ R^d1."*
>
> — §Feature extraction

**Table 10 — Feature domains:**

| Domain | Features per channel |
|--------|---------------------|
| Spectral (Welch PSD) | Log power in Delta, Theta, Alpha, Beta, Gamma |
| Wavelet (db4, level 5) | Mean and Std of coefficients (level-wise) |
| Statistical | Mean, Std, Skewness, Kurtosis |
| Entropy | Permutation Entropy |

**Eq. 8 (log bandpower):**
$$P_{band} = \log\left(\int_{f_1}^{f_2} P(f)\,df + \varepsilon\right)$$

**Eq. 11 (permutation entropy):**
$$H_{perm} = -\sum_{k=1}^{m!} p_k \log p_k$$

In [ ]:
# --- Spectral features (Eq. 8) ---
from scipy.signal import welch as scipy_welch

def bandpower(psd, freqs, fmin, fmax):
    """§Feature extraction, Eq. 8"""
    idx = np.logical_and(freqs >= fmin, freqs <= fmax)
    eps = 1e-10  # [UNSPECIFIED] stabilization constant
    return float(np.log(np.trapz(psd[idx], freqs[idx]) + eps))

BANDS = {'delta':(0.5,4), 'theta':(4,7), 'alpha':(7,13), 'beta':(13,40), 'gamma':(40,45)}
FS = 500.0  # §Data acquisition — 500 Hz

# Toy EEG segment: (L, C)
seg = np.random.randn(L, C)

spectral_feats = []
for c in range(C):
    freqs, psd = scipy_welch(seg[:, c], fs=FS, nperseg=min(32, L))  # small nperseg for toy data
    for (fmin, fmax) in BANDS.values():
        spectral_feats.append(bandpower(psd, freqs, fmin, fmax))
spectral_feats = np.array(spectral_feats)

assert spectral_feats.shape == (C * 5,), f'Expected {C*5}, got {spectral_feats.shape}'
print(f'✓ Spectral features: shape {spectral_feats.shape} (C={C} channels × 5 bands)')

In [ ]:
# --- Wavelet features (Eq. 9) ---
import pywt

wavelet_feats = []
for c in range(C):
    # §Feature extraction — "Daubechies-4 (db4) wavelets up to level 5"
    coeffs = pywt.wavedec(seg[:, c], wavelet='db4', level=min(5, pywt.dwt_max_level(L, 'db4')))
    for wk in coeffs:  # Eq. 9: mean and std of each coefficient set
        wavelet_feats.extend([np.mean(wk), np.std(wk, ddof=1) if len(wk)>1 else 0.0])
wavelet_feats = np.array(wavelet_feats)

print(f'✓ Wavelet features: shape {wavelet_feats.shape}')

In [ ]:
# --- Statistical features (Eq. 10) ---
from scipy.stats import skew, kurtosis

stat_feats = []
for c in range(C):
    x = seg[:, c]
    # §Feature extraction, Eq. 10 — mean, std, skewness (γ), kurtosis (κ)
    stat_feats.extend([np.mean(x), np.std(x, ddof=1), float(skew(x)), float(kurtosis(x))])
stat_feats = np.array(stat_feats)

assert stat_feats.shape == (C * 4,)
print(f'✓ Statistical features: shape {stat_feats.shape} (C={C} × [mean,std,skew,kurt])')

In [ ]:
# --- Permutation entropy (Eq. 11) ---
def permutation_entropy(x, order=3):
    """§Feature extraction, Eq. 11 — H_perm = -Σ p_k log p_k
    [UNSPECIFIED] order m not stated; using m=3
    """
    n = len(x)
    if n < order:
        return 0.0
    patterns = [tuple(np.argsort(x[i:i+order])) for i in range(n - order + 1)]
    counts = Counter(patterns)
    total = sum(counts.values())
    return -sum((c/total)*math.log(c/total) for c in counts.values())

entropy_feats = np.array([permutation_entropy(seg[:, c]) for c in range(C)])

# --- Full handcrafted vector ---
f_hand = np.concatenate([spectral_feats, wavelet_feats, stat_feats, entropy_feats])
d1 = len(f_hand)

print(f'✓ Entropy features: shape {entropy_feats.shape}')
print(f'✓ Full handcrafted vector f_hand: shape ({d1},) — paper expects d1 ≈ 250-300')

---
## 2. Automated Feature Extraction: 1D-CNN (§Automated feature extraction, Eq. 12)

> *"We propose a lightweight 1D Convolutional Neural Network (1D-CNN) for automated
> detection of features from multichannel EEG signals. It consists of three
> convolutional layers followed by max pooling and a global average pooling layer
> for dimensionality reduction."*
>
> — §Automated feature extraction

**Eq. 12:**
$$h^{(l)} = \text{ReLU}\left(\text{BN}\left(W^{(l)} * h^{(l-1)} + b^{(l)}\right)\right)$$

**Table 11** — 1D-CNN achieves 94.27% accuracy vs 2D-CNN 91.83% at lower computational cost.  
**Table 12** — CNN output dimension d₂ = 256.

In [ ]:
class ConvBlock1D(nn.Module):
    """§Automated feature extraction, Eq. 12 — Conv1D → BN → ReLU → MaxPool → Dropout"""
    def __init__(self, in_ch, out_ch, kernel=5, dropout=0.3):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel, padding=kernel//2)
        self.bn   = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)   # [UNSPECIFIED] kernel size; using 2
        self.drop = nn.Dropout(dropout)

    def forward(self, x):  # (B, in_ch, T)
        return self.drop(self.pool(self.relu(self.bn(self.conv(x)))))  # (B, out_ch, T//2)


class EEG1DCNN(nn.Module):
    """§Automated feature extraction — 3-block 1D-CNN → GlobalAvgPool → d2"""
    def __init__(self, n_channels, output_dim, filters=(32,64,128), kernel=5, dropout=0.3):
        super().__init__()
        self.block1 = ConvBlock1D(n_channels, filters[0], kernel, dropout)
        self.block2 = ConvBlock1D(filters[0], filters[1], kernel, dropout)
        self.block3 = ConvBlock1D(filters[1], filters[2], kernel, dropout)
        self.gap   = nn.AdaptiveAvgPool1d(1)  # §Automated feature extraction — global avg pool
        self.proj  = nn.Linear(filters[2], output_dim)  # → d2=256

    def forward(self, x):            # (B, C, L)
        x = self.block1(x)            # (B, 32, L//2)
        x = self.block2(x)            # (B, 64, L//4)
        x = self.block3(x)            # (B, 128, L//8)
        x = self.gap(x).squeeze(-1)  # (B, 128)
        return self.proj(x)           # (B, d2)


# Use toy dimensions: C=4 channels, d2=D2=32
cnn = EEG1DCNN(n_channels=C, output_dim=D2, filters=(8,16,32), kernel=5)
seg_tensor = torch.randn(BATCH, C, L)  # (B, C, L)
f_cnn = cnn(seg_tensor)                # (B, d2)

assert f_cnn.shape == (BATCH, D2), f'Expected ({BATCH},{D2}), got {f_cnn.shape}'
print(f'✓ 1D-CNN: input {seg_tensor.shape} → embedding {f_cnn.shape}')
print(f'  Parameters: {sum(p.numel() for p in cnn.parameters()):,} (paper full model: 0.94M total)')

---
## 3. Feature Fusion (§Feature extraction, Eq. 13–16)

> *"To combine domain-level and learned representations, handcrafted and CNN-derived
> features were concatenated:*
> $$f^{(i,j)} = [f_{hand}^{(i,j)} \|\, f_{cnn}^{(i,j)}] \in \mathbb{R}^{d_1+d_2}$$
>
> — §Feature extraction, Eq. 13

**Then (§Key contributions):**
1. PCC-based removal of highly correlated handcrafted features
2. PSO-based wrapper feature selection  ← [UNSPECIFIED hyperparameters — stub in code]
3. SMOTE oversampling (Eq. 14) — applied **within training folds only**
4. PCA (99% variance retained, Eq. 15–16) → dfinal ≈ 60–80

**Table 12:**

| Stage | Dimensionality |
|-------|---------------|
| Handcrafted | ≈ 250–300 |
| CNN (d₂) | 256 |
| Concatenated | ≈ 550 |
| Post-PCA (d_final) | ≈ 60–80 |

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Simulate a small dataset of fused feature vectors
N_SEGS = 40  # toy: 40 EEG segments
D1_TOY = 30  # toy d1 (paper: ≈250-300)
D2_TOY = D2  # toy d2

# §Feature extraction, Eq. 13 — concatenation
F_hand_all = np.random.randn(N_SEGS, D1_TOY)  # handcrafted features
F_cnn_all  = np.random.randn(N_SEGS, D2_TOY)  # CNN embeddings
F_fused    = np.concatenate([F_hand_all, F_cnn_all], axis=1)  # (N, d1+d2)
assert F_fused.shape == (N_SEGS, D1_TOY + D2_TOY)
print(f'✓ Eq. 13 fusion: {F_hand_all.shape} + {F_cnn_all.shape} → {F_fused.shape}')

# PCC-based feature removal
corr = np.corrcoef(F_fused.T)  # (n_features, n_features)
keep = []
dropped = set()
for i in range(F_fused.shape[1]):
    if i in dropped: continue
    keep.append(i)
    for j in range(i+1, F_fused.shape[1]):
        if abs(corr[i,j]) > 0.95:   # [UNSPECIFIED] threshold=0.95
            dropped.add(j)
F_pcc = F_fused[:, keep]
print(f'✓ PCC selection: {F_fused.shape[1]} → {F_pcc.shape[1]} features')

# §Feature extraction, Eq. 15-16 — PCA (99% variance)
# Note: fit on training data only; here we use all toy data for demonstration
scaler = StandardScaler()
F_scaled = scaler.fit_transform(F_pcc)
pca = PCA(n_components=0.99)  # §Feature extraction — "retaining 99% of variance"
F_final = pca.fit_transform(F_scaled)
print(f'✓ PCA (99% variance): {F_pcc.shape[1]} → {F_final.shape[1]} components (dfinal)')
print(f'  Paper expects dfinal ≈ 60-80 (Table 12) on full 550-dim input')

---
## 4. NeuroFusionNet Classifier (§Classification network, Table 15, Fig. 7)

> *"The DNN consists of five fully connected (dense) layers, each followed by batch
> normalization (BN), dropout, and a non-linear activation. A skip connection between
> the first and third dense layers facilitates residual learning and promotes smooth
> gradient propagation."*
>
> — §Classification network

**Eq. 18 (dense layer transform):**
$$h^{(l)} = f\left(W^{(l)} h^{(l-1)} + b^{(l)}\right), \quad f = \text{LeakyReLU}$$

**Eq. 19 (output):**
$$\hat{y} = \text{softmax}\left(W^{(L)} h^{(L-1)} + b^{(L)}\right)$$

**Table 15 architecture:**

| Layer | Operation | Details |
|-------|-----------|--------|
| Input | Feature Vector | dfinal ≈ 60–80 |
| Dense-1 | Dense + LeakyReLU + Dropout | 512 units, p=0.4 |
| Dense-2 | Skip Connection | 256 units (ReLU) |
| Dense-3 | Dense + Add + Dropout | 256 units + skip, p=0.4 |
| Dense-4 | Dense + BN + Dropout | 128 units, p=0.3 |
| Dense-5 | Dense + BN + Dropout | 64 units, p=0.3 |
| Output | Dense + Softmax | 3 classes |

In [ ]:
class NeuroFusionNetClassifier(nn.Module):
    """§Classification network, Table 15, Fig. 7 — 5-layer DNN with skip connection."""

    def __init__(self, input_dim, n_classes=3,
                 units=(512,256,256,128,64),
                 dropouts=(0.4, 0.4, 0.3, 0.3),   # Table 14: p1=p2=0.4, p3=p4=0.3
                 leaky_slope=0.01):                # [UNSPECIFIED] slope; using PyTorch default
        super().__init__()
        d1, d2, d3, d4, d5 = units

        # Dense-1: §Table 15 — LeakyReLU + Dropout, 512 units
        self.dense1 = nn.Linear(input_dim, d1)
        self.act1   = nn.LeakyReLU(leaky_slope)   # §Eq. 18 — LeakyReLU
        self.drop1  = nn.Dropout(dropouts[0])      # Table 14: p1=0.4

        # Dense-2: §Table 15 — ReLU, 256 units (skip connection source)
        self.dense2 = nn.Linear(d1, d2)
        self.act2   = nn.ReLU()                    # Table 15 explicitly: ReLU for Dense-2

        # Skip projection: Dense-1 (d1) → Dense-3 size (d3)
        # [UNSPECIFIED] exact skip mechanism; using linear projection
        self.skip_proj = nn.Linear(d1, d3)

        # Dense-3: §Table 15 — Dense + Add(skip) + Dropout, 256 units
        self.dense3 = nn.Linear(d2, d3)
        self.act3   = nn.LeakyReLU(leaky_slope)
        self.drop3  = nn.Dropout(dropouts[1])      # Table 14: p2=0.4

        # Dense-4: §Table 15 — Dense + BN + Dropout, 128 units
        self.dense4 = nn.Linear(d3, d4)
        self.bn4    = nn.BatchNorm1d(d4)
        self.act4   = nn.LeakyReLU(leaky_slope)
        self.drop4  = nn.Dropout(dropouts[2])      # Table 14: p3=0.3

        # Dense-5: §Table 15 — Dense + BN + Dropout, 64 units
        self.dense5 = nn.Linear(d4, d5)
        self.bn5    = nn.BatchNorm1d(d5)
        self.act5   = nn.LeakyReLU(leaky_slope)
        self.drop5  = nn.Dropout(dropouts[3])      # Table 14: p4=0.3

        # Output: §Table 15 — Dense + Softmax, 3 classes
        self.output = nn.Linear(d5, n_classes)

    def forward_logits(self, x):    # (B, dfinal)
        """Returns raw logits for use with nn.CrossEntropyLoss."""
        # §Eq. 17 — x^(0) = f_final^(i)
        h1 = self.drop1(self.act1(self.dense1(x)))    # (B, 512)
        h2 = self.act2(self.dense2(h1))                # (B, 256) ← skip source
        skip = self.skip_proj(h1)                       # (B, 256) ← projected skip
        # §Fig. 7 — residual addition at Dense-3
        h3 = self.drop3(self.act3(self.dense3(h2) + skip))  # (B, 256)
        h4 = self.drop4(self.act4(self.bn4(self.dense4(h3))))  # (B, 128)
        h5 = self.drop5(self.act5(self.bn5(self.dense5(h4))))  # (B, 64)
        return self.output(h5)                          # (B, n_classes)

    def forward(self, x):           # §Eq. 19 — ŷ = softmax(...)
        return F.softmax(self.forward_logits(x), dim=-1)


# Toy: dfinal=DFINAL=20, reduced layer sizes for speed
TOY_UNITS = (32, 16, 16, 8, 8)  # paper: (512, 256, 256, 128, 64)
model = NeuroFusionNetClassifier(DFINAL, n_classes=N_CLS, units=TOY_UNITS)

x = torch.randn(BATCH, DFINAL)
probs = model(x)

assert probs.shape == (BATCH, N_CLS), f'Expected ({BATCH},{N_CLS}), got {probs.shape}'
assert torch.allclose(probs.sum(dim=-1), torch.ones(BATCH), atol=1e-5), 'Probabilities must sum to 1'
print(f'✓ Classifier forward: input {x.shape} → probabilities {probs.shape}')
print(f'  Prob sums: {probs.sum(dim=-1).tolist()} (should be ≈1.0)')
print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')

---
## 5. Loss Function (§Classification network, Eq. 20)

> *"The network is trained by minimizing the categorical cross-entropy loss with
> L2 weight regularization:"*
>
> $$\mathcal{L} = -\sum_{k=1}^{K} y_k \log \hat{y}_k + \lambda \sum_{l=1}^{L} \|W^{(l)}\|_2^2$$
>
> — §Classification network, Eq. 20

**Table 14:** λ = 0.001 (set as `weight_decay` in Adam optimizer, not manually added to loss).

In [ ]:
# §Eq. 20 — Cross-entropy term
# nn.CrossEntropyLoss expects LOGITS (pre-softmax)
criterion = nn.CrossEntropyLoss()

logits = model.forward_logits(x)           # (B, n_classes)
targets = torch.randint(0, N_CLS, (BATCH,))  # (B,)

loss = criterion(logits, targets)
print(f'✓ Loss (cross-entropy): {loss.item():.4f}')
print(f'  L2 regularization (λ=0.001) is handled by optimizer weight_decay, not added here')

# Verify: Adam optimizer with weight_decay = λ = 0.001 (Table 14)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4,          # Table 14 — Initial Learning Rate
    weight_decay=0.001  # Table 14 — L2 (λ = 0.001)
    # [UNSPECIFIED] betas/eps — using PyTorch defaults (0.9, 0.999, 1e-8)
)
from torch.optim.lr_scheduler import ReduceLROnPlateau
scheduler = ReduceLROnPlateau(
    optimizer, mode='min',
    factor=0.5,   # Table 14 — factor = 0.5
    patience=10   # Table 14 — patience = 10
)
print(f'✓ Optimizer: Adam lr={optimizer.param_groups[0]["lr"]}, '
      f'weight_decay={optimizer.param_groups[0]["weight_decay"]}')

---
## 6. Training Step — Gradient Flow Check

In [ ]:
# One forward + backward pass to verify gradient flow
model.train()
optimizer.zero_grad()
loss = criterion(model.forward_logits(x), targets)
loss.backward()
optimizer.step()

print('Gradient check:')
no_grad = []
for name, param in model.named_parameters():
    if param.grad is None:
        no_grad.append(name)
    else:
        print(f'  ✓ {name}: grad_norm={param.grad.norm():.4f}')

if no_grad:
    print(f'\n⚠ No gradient for: {no_grad}')
else:
    print('\n✓ All parameters have gradients — skip connection is active')

---
## 7. Full Pipeline Integration

The complete NeuroFusionNet pipeline:

```
Raw EEG  →  Preprocessing (bandpass→ASR→ICA→z-score→segment)
         →  Handcrafted features (spectral+wavelet+stat+entropy)
         →  CNN features (1D-CNN → 256-dim)
         →  Fusion (concat → PCC → PSO → SMOTE → PCA)
         →  NeuroFusionNet DNN (512→256→256→128→64→3)
         →  AD / FTD / CN
```

In [ ]:
# End-to-end forward pass with toy data
# (Skips preprocessing and PCA — uses random tensors at each stage)

# Stage 1: Simulate fused features (post-PCA)
toy_features = torch.randn(BATCH, DFINAL)  # (B, dfinal)

# Stage 2: Classifier forward pass
model.eval()
with torch.no_grad():
    probs = model(toy_features)              # (B, 3)
    pred_classes = probs.argmax(dim=-1)      # (B,)

CLASS_NAMES = ['AD', 'FTD', 'CN']
for b in range(BATCH):
    print(f'  Sample {b}: probs={probs[b].tolist()} → predicted={CLASS_NAMES[pred_classes[b].item()]}')

# Verify output properties
assert probs.shape == (BATCH, 3)
assert (probs >= 0).all() and (probs <= 1).all()
assert torch.allclose(probs.sum(dim=-1), torch.ones(BATCH), atol=1e-5)
print('\n✓ Full pipeline sanity checks passed')

---
## 8. Common Pitfalls

Based on the paper's description and ambiguity audit:

### 1. SMOTE must only be applied within training folds
The paper is explicit: *"SMOTE was applied strictly within training folds during cross-validation; validation and test folds remained unchanged."* Applying SMOTE before splitting will cause data leakage and inflated metrics.

### 2. Use logits, not probabilities, with CrossEntropyLoss
`nn.CrossEntropyLoss` expects raw logits (pre-softmax). Use `model.forward_logits()` during training, not `model.forward()`. Passing softmax outputs to CrossEntropyLoss applies softmax twice, silently producing wrong gradients.

### 3. PSO hyperparameters are unspecified — this is a major source of variance
The PSO stub raises `NotImplementedError`. You must configure PSO separately. Different swarm/iteration choices will produce different feature subsets. See `REPRODUCTION_NOTES.md §1` for guidance.

### 4. PCA must be fit only on training data
Fit `StandardScaler` + `PCA` only on training segments, then `transform()` validation and test segments. Fitting on all data introduces leakage and will report optimistic variance retention.

### 5. dfinal varies per run
The PCA output dimensionality (dfinal ≈ 60–80) depends on the training data. You must set `ClassifierConfig.input_dim = pca.n_components_` after fitting PCA. A mismatch causes a shape error at the first Dense-1 layer.

### 6. Subject-level (not segment-level) stratified splits
Split by *subject*, not by segment. If you split by segment, EEG segments from the same subject appear in both train and test, causing severe data leakage (the model memorizes individual brain signatures).

### 7. Dense-2 uses ReLU, not LeakyReLU
Table 15 says Dense-2 is a "Skip Connection" with "ReLU". All other dense layers use LeakyReLU (Eq. 18). This asymmetry is easy to miss.

### 8. ε in bandpower (Eq. 8) — silent NaN source
If a frequency band has zero power (e.g., gamma in very short segments), `log(0)` = -inf. The ε stabilization constant prevents this but its value is unspecified. Using `eps=1e-10` is safe for typical EEG power ranges.